# CocinaAI — EDA e Ingeniería de Features v3

**Proyecto:** CocinaAI — Aprovecha tu alacena con recetas mexicanas  
**Fuentes de datos:** Spoonacular API + TheMealDB API  

Este notebook cubre:
1. Ingesta desde dos fuentes (Spoonacular + TheMealDB)
2. Limpieza, normalización y fusión de datasets
3. Ingeniería de variables (dificultad, alacena, TF-IDF)
4. Análisis exploratorio y visualizaciones
5. Exportación para el modelo de clustering

### Imports

In [ ]:
!pip install requests pandas numpy plotly scikit-learn -q

In [ ]:
import json
import time
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import scipy.sparse
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [ ]:
# Colab only
from google.colab import userdata
spoonacular_api = userdata.get('SPOONACULAR_API')

### Variables globales

In [ ]:
# Spoonacular: ajusta según tu cuota diaria disponible
# Con plan free (150 pts/día) y nutrition=True, cada llamada cuesta ~5 pts
# Recomendado: 50 recetas con nutrición, o 100 sin nutrición
N_SPOONACULAR = 50

# TheMealDB: sin límite, gratis, sin API key
# Descarga todas las recetas mexicanas disponibles (~30-50)
# Más: categorías relacionadas (Chicken, Beef, Vegetarian, etc.)
THEMEALDB_AREAS = ['Mexican']
THEMEALDB_CATEGORIES = ['Chicken', 'Beef', 'Pork', 'Vegetarian', 'Seafood', 'Lamb']


## 1. Ingesta de datos

### 1.1 Spoonacular API

Fuente principal con datos estructurados de recetas mexicanas. Usamos `addRecipeNutrition=False` para conservar puntos de cuota y obtener más recetas.

In [ ]:
def get_spoonacular_recipes(cuisine: str, number: int, api_key: str) -> list:
    """
    Obtiene recetas desde Spoonacular API.
    addRecipeNutrition=False para ahorrar puntos de cuota.
    """
    url = 'https://api.spoonacular.com/recipes/complexSearch'
    params = {
        'cuisine': cuisine,
        'number': number,
        'addRecipeInformation': True,
        'addRecipeNutrition': False,
        'apiKey': api_key
    }
    resp = requests.get(url, params=params, timeout=15)
    return resp.json().get('results', [])


In [ ]:
def parse_spoonacular(recipe: dict) -> dict:
    """
    Normaliza una receta de Spoonacular al formato común del proyecto.
    """
    ingredients = recipe.get('extendedIngredients', [])
    ingredient_names = [i.get('name', '').lower().strip() for i in ingredients
                        if i.get('name')]

    return {
        'id': f"sp_{recipe.get('id')}",
        'titulo': recipe.get('title', ''),
        'fuente': 'Spoonacular',
        'tiempo_minutos': recipe.get('readyInMinutes'),
        'porciones': recipe.get('servings'),
        'num_ingredientes': len(ingredient_names),
        'ingredientes': ingredient_names,
        'vegana': recipe.get('vegan', False),
        'vegetariana': recipe.get('vegetarian', False),
        'sin_gluten': recipe.get('glutenFree', False),
        'url': recipe.get('sourceUrl', ''),
        'imagen': recipe.get('image', ''),
    }


### 1.2 TheMealDB API

Fuente complementaria, completamente gratuita y sin límite de peticiones. Tiene menor estructura nutricional pero mayor cobertura de recetas mexicanas tradicionales. Descargamos por área geográfica (Mexican) y por categorías de proteína.

In [ ]:
def get_themealdb_by_area(area: str) -> list:
    """
    Obtiene lista de recetas de TheMealDB filtradas por área geográfica.
    """
    url = f'https://www.themealdb.com/api/json/v1/1/filter.php?a={area}'
    resp = requests.get(url, timeout=15)
    return resp.json().get('meals', []) or []

def get_themealdb_by_category(category: str) -> list:
    """
    Obtiene lista de recetas de TheMealDB filtradas por categoría de platillo.
    """
    url = f'https://www.themealdb.com/api/json/v1/1/filter.php?c={category}'
    resp = requests.get(url, timeout=15)
    return resp.json().get('meals', []) or []

def get_themealdb_detail(meal_id: str) -> dict:
    """
    Obtiene el detalle completo de una receta por su ID.
    """
    url = f'https://www.themealdb.com/api/json/v1/1/lookup.php?i={meal_id}'
    resp = requests.get(url, timeout=15)
    meals = resp.json().get('meals', [])
    return meals[0] if meals else {}


In [ ]:
def parse_themealdb(meal: dict) -> dict:
    """
    Normaliza una receta de TheMealDB al formato común del proyecto.
    TheMealDB guarda ingredientes en columnas strIngredient1..20.
    """
    ingredients = []
    for i in range(1, 21):
        ing = meal.get(f'strIngredient{i}', '')
        if ing and ing.strip():
            ingredients.append(ing.lower().strip())

    category = meal.get('strCategory', '').lower()
    tags = (meal.get('strTags') or '').lower()
    is_vegetarian = any(v in category or v in tags
                        for v in ['vegetarian', 'vegan', 'veggie'])
    is_vegan = 'vegan' in category or 'vegan' in tags

    return {
        'id': f"mdb_{meal.get('idMeal')}",
        'titulo': meal.get('strMeal', ''),
        'fuente': 'TheMealDB',
        'tiempo_minutos': None,  # TheMealDB no provee tiempo
        'porciones': None,
        'num_ingredientes': len(ingredients),
        'ingredientes': ingredients,
        'vegana': is_vegan,
        'vegetariana': is_vegetarian,
        'sin_gluten': False,
        'url': meal.get('strSource', ''),
        'imagen': meal.get('strMealThumb', ''),
    }


### 1.3 Descarga de datos

In [ ]:
# ── Spoonacular ──
print('Descargando desde Spoonacular...')
raw_spoonacular = get_spoonacular_recipes('mexican', N_SPOONACULAR, spoonacular_api)
spoonacular_list = [parse_spoonacular(r) for r in raw_spoonacular]
print(f'  Spoonacular: {len(spoonacular_list)} recetas')

In [ ]:
# ── TheMealDB por área mexicana ──
print('Descargando desde TheMealDB (area Mexican)...')
mealdb_ids = set()
mealdb_raw = []

for area in THEMEALDB_AREAS:
    meals_list = get_themealdb_by_area(area)
    for m in meals_list:
        if m['idMeal'] not in mealdb_ids:
            mealdb_ids.add(m['idMeal'])
            detail = get_themealdb_detail(m['idMeal'])
            if detail:
                mealdb_raw.append(parse_themealdb(detail))
            time.sleep(0.1)  # evitamos rate limiting

print(f'  TheMealDB (Mexican): {len(mealdb_raw)} recetas')

In [ ]:
# ── TheMealDB por categorías (amplía el dataset) ──
print('Descargando desde TheMealDB (categorias)...')
MAX_PER_CATEGORY = 15  # limitamos para no sesgar el dataset

for category in THEMEALDB_CATEGORIES:
    meals_list = get_themealdb_by_category(category)[:MAX_PER_CATEGORY]
    added = 0
    for m in meals_list:
        if m['idMeal'] not in mealdb_ids:
            mealdb_ids.add(m['idMeal'])
            detail = get_themealdb_detail(m['idMeal'])
            if detail:
                mealdb_raw.append(parse_themealdb(detail))
                added += 1
            time.sleep(0.1)
    print(f'  {category}: +{added} recetas')

print(f'Total TheMealDB: {len(mealdb_raw)} recetas')

In [ ]:
# ── Fusión de datasets ──
df_spoonacular = pd.DataFrame(spoonacular_list)
df_mealdb = pd.DataFrame(mealdb_raw)

recipes_df = pd.concat([df_spoonacular, df_mealdb], ignore_index=True)
print(f'Dataset combinado: {recipes_df.shape[0]} recetas totales')
print()
print('Por fuente:')
print(recipes_df['fuente'].value_counts())

## 2. Limpieza y calidad de datos

Las dos fuentes tienen estructuras distintas. Documentamos cada decisión de limpieza.

In [ ]:
print('Valores nulos por columna:')
print(recipes_df.isnull().sum())
print()
print('Tipos de datos:')
print(recipes_df.dtypes)

In [ ]:
# Eliminamos recetas sin ingredientes (no se pueden vectorizar)
n_antes = len(recipes_df)
recipes_df = recipes_df[recipes_df['num_ingredientes'] > 0]
recipes_df = recipes_df[recipes_df['ingredientes'].apply(len) > 0]
print(f'Eliminadas por sin ingredientes: {n_antes - len(recipes_df)}')

# Eliminamos duplicados por título (pueden venir de ambas fuentes)
n_antes = len(recipes_df)
recipes_df = recipes_df.drop_duplicates(subset='titulo', keep='first')
print(f'Eliminadas por titulo duplicado: {n_antes - len(recipes_df)}')

# Outliers de tiempo (solo aplica a Spoonacular, TheMealDB tiene None)
mask_tiempo = recipes_df['tiempo_minutos'].notna()
n_antes = len(recipes_df)
recipes_df = recipes_df[~mask_tiempo | (recipes_df['tiempo_minutos'] <= 480)]
print(f'Eliminadas por tiempo > 480 min: {n_antes - len(recipes_df)}')

recipes_df = recipes_df.reset_index(drop=True)
print(f'\nRecetas limpias: {len(recipes_df)}')

In [ ]:
# Tiempo promedio para imputar TheMealDB (donde es None)
tiempo_promedio = recipes_df['tiempo_minutos'].median()
recipes_df['tiempo_minutos'] = recipes_df['tiempo_minutos'].fillna(tiempo_promedio)
recipes_df['porciones'] = recipes_df['porciones'].fillna(4)

print(f'Tiempo mediano imputado para TheMealDB: {tiempo_promedio} min')
print('Justificación: usamos mediana (robusta a outliers) en lugar de media.')

In [ ]:
recipes_df[['titulo','fuente','tiempo_minutos','num_ingredientes']].head(10)

## 3. Ingeniería de variables

### 3.1 Dificultad

Score compuesto basado en número de ingredientes y tiempo de preparación.

In [ ]:
def clasificar_dificultad(num_ing, tiempo):
    score = 0
    if num_ing > 12: score += 2
    elif num_ing > 7: score += 1
    if tiempo > 60: score += 2
    elif tiempo > 30: score += 1
    if score <= 1: return 'Facil'
    elif score <= 3: return 'Media'
    else: return 'Dificil'

recipes_df['dificultad'] = recipes_df.apply(
    lambda r: clasificar_dificultad(r['num_ingredientes'], r['tiempo_minutos']), axis=1
)
recipes_df['dificultad'].value_counts()

### 3.2 Score de aprovechamiento de alacena

Métrica central del proyecto. Mide qué tan preparable es una receta con una despensa mexicana típica. Se construye con 3 componentes:

- `prop_alacena`: proporción de ingredientes en el top 60 más frecuentes del dataset
- `pocos_ingredientes`: 1 si tiene 7 o menos ingredientes (más fácil de tener todo)
- `es_rapida`: 1 si toma 35 minutos o menos

Pesos: 60% alacena + 25% pocos ingredientes + 15% rapidez

In [ ]:
# Ingredientes más comunes en todo el dataset combinado
all_ingredients = [ing for sublist in recipes_df['ingredientes'] for ing in sublist]
ingredient_counts = Counter(all_ingredients)
top_60_alacena = set([ing for ing, _ in ingredient_counts.most_common(60)])

print(f'Total ingredientes únicos en el dataset: {len(ingredient_counts)}')
print(f'Top 60 ingredientes de alacena:')
print(', '.join(list(top_60_alacena)[:20]), '...')

In [ ]:
# Proporción de ingredientes que están en la alacena típica
recipes_df['prop_alacena'] = recipes_df['ingredientes'].apply(
    lambda ings: round(len(set(ings) & top_60_alacena) / max(len(ings), 1), 3)
)

# Receta con pocos ingredientes (más fácil de tener todo en casa)
recipes_df['pocos_ingredientes'] = (recipes_df['num_ingredientes'] <= 7).astype(int)

# Receta rápida
recipes_df['es_rapida'] = (recipes_df['tiempo_minutos'] <= 35).astype(int)

# Score de aprovechamiento compuesto
recipes_df['score_alacena'] = (
    0.60 * recipes_df['prop_alacena'] +
    0.25 * recipes_df['pocos_ingredientes'] +
    0.15 * recipes_df['es_rapida']
).round(3)

print('Score alacena — estadísticas:')
print(recipes_df['score_alacena'].describe().round(3))

In [ ]:
# Top 10 recetas más aprovechables
recipes_df[['titulo','fuente','num_ingredientes','prop_alacena',
            'score_alacena','dificultad']].sort_values(
    'score_alacena', ascending=False).head(10)

### 3.3 Vectorización TF-IDF de ingredientes

Convertimos cada receta en un vector numérico basado en sus ingredientes. TF-IDF asigna mayor peso a ingredientes distintivos de cada receta. Este vector es el input del modelo de clustering.

In [ ]:
# Convertimos lista de ingredientes a texto
recipes_df['ingredientes_texto'] = recipes_df['ingredientes'].apply(
    lambda ings: ' '.join([i.replace(' ', '_') for i in ings if i])
)

# TF-IDF con hasta 300 features
tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2
)
tfidf_matrix = tfidf.fit_transform(recipes_df['ingredientes_texto'])

print(f'Matriz TF-IDF: {tfidf_matrix.shape}')
print(f'  {tfidf_matrix.shape[0]} recetas x {tfidf_matrix.shape[1]} features')

In [ ]:
# Ingredientes con mayor peso TF-IDF
feature_names = tfidf.get_feature_names_out()
avg_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_tfidf_df = pd.DataFrame({'ingrediente': feature_names, 'peso': avg_tfidf})\
               .sort_values('peso', ascending=False).head(25)
top_tfidf_df

### 3.4 Normalización

In [ ]:
# Normalizamos variables numéricas para clustering
cols_norm = ['num_ingredientes', 'tiempo_minutos', 'score_alacena',
             'prop_alacena', 'pocos_ingredientes', 'es_rapida']

scaler = MinMaxScaler()
norm_vals = scaler.fit_transform(recipes_df[cols_norm].fillna(0))
norm_df = pd.DataFrame(norm_vals,
                       columns=[f'{c}_norm' for c in cols_norm],
                       index=recipes_df.index)
recipes_df = pd.concat([recipes_df, norm_df], axis=1)
print('Columnas normalizadas agregadas.')

## 4. Análisis exploratorio

### 4.1 Cobertura del dataset

In [ ]:
fig = px.pie(
    recipes_df, names='fuente',
    title='Distribución de recetas por fuente',
    color_discrete_map={'Spoonacular': '#B5722A', 'TheMealDB': '#1D9E75'},
    template='plotly_white'
)
fig.show()

print(recipes_df.groupby('fuente').agg(
    recetas=('id','count'),
    ing_promedio=('num_ingredientes','mean'),
    tiempo_promedio=('tiempo_minutos','mean')
).round(1))

### 4.2 Ingredientes más frecuentes

In [ ]:
top_ingredients_df = pd.DataFrame(
    ingredient_counts.most_common(25), columns=['ingrediente','frecuencia']
)

fig = go.Figure(go.Bar(
    x=top_ingredients_df['ingrediente'],
    y=top_ingredients_df['frecuencia'],
    marker_color='#B5722A'
))
fig.update_layout(
    title='Top 25 ingredientes más frecuentes (Spoonacular + TheMealDB)',
    xaxis_tickangle=-45, template='plotly_white', height=420
)
fig.show()

### 4.3 Distribución de dificultad y tiempo

In [ ]:
fig1 = px.histogram(
    recipes_df, x='tiempo_minutos', nbins=25, color='fuente',
    title='Distribución de tiempos de preparación por fuente',
    color_discrete_map={'Spoonacular': '#B5722A', 'TheMealDB': '#1D9E75'},
    barmode='overlay', template='plotly_white'
)
fig1.update_traces(opacity=0.75)
fig1.show()

In [ ]:
dif_src = recipes_df.groupby(['dificultad','fuente']).size().reset_index(name='count')
fig2 = px.bar(
    dif_src, x='dificultad', y='count', color='fuente', barmode='group',
    title='Distribución de dificultad por fuente',
    color_discrete_map={'Spoonacular': '#B5722A', 'TheMealDB': '#1D9E75'},
    template='plotly_white'
)
fig2.show()

### 4.4 Score de alacena

In [ ]:
fig = px.histogram(
    recipes_df, x='score_alacena', nbins=20, color='fuente',
    title='Distribución del Score de Aprovechamiento de Alacena',
    color_discrete_map={'Spoonacular': '#B5722A', 'TheMealDB': '#1D9E75'},
    barmode='overlay', template='plotly_white'
)
fig.add_vline(x=recipes_df['score_alacena'].mean(), line_dash='dash',
              line_color='#534AB7', annotation_text='Promedio')
fig.update_traces(opacity=0.75)
fig.show()

In [ ]:
# Score por dificultad
sc_dif = recipes_df.groupby('dificultad')['score_alacena'].mean().round(3).reset_index()
fig = go.Figure(go.Bar(
    x=sc_dif['dificultad'], y=sc_dif['score_alacena'],
    marker_color=['#1D9E75','#EF9F27','#D85A30']
))
fig.update_layout(
    title='Score de alacena promedio por dificultad',
    template='plotly_white'
)
fig.show()

In [ ]:
# Scatter: num_ingredientes vs score_alacena
fig = px.scatter(
    recipes_df, x='num_ingredientes', y='score_alacena',
    color='dificultad', hover_name='titulo', symbol='fuente',
    title='Número de ingredientes vs Score de alacena',
    template='plotly_white',
    color_discrete_map={'Facil':'#1D9E75','Media':'#EF9F27','Dificil':'#D85A30'}
)
fig.show()

### 4.5 Tabla resumen

In [ ]:
resumen = recipes_df.groupby(['fuente','dificultad']).agg(
    recetas=('id','count'),
    tiempo_prom=('tiempo_minutos','mean'),
    ingredientes_prom=('num_ingredientes','mean'),
    score_alacena_prom=('score_alacena','mean')
).round(2).reset_index()
resumen

In [ ]:
print('Top 10 recetas más aprovechables del dataset combinado:')
recipes_df[['titulo','fuente','num_ingredientes','tiempo_minutos',
            'score_alacena','dificultad']].sort_values(
    'score_alacena', ascending=False).head(10)

## 5. Exportación

Tres archivos para el notebook de clustering:
- `recipes_df.csv` — dataset limpio con todas las features
- `tfidf_matrix.npz` — matriz TF-IDF esparsa
- `tfidf_feature_names.json` — nombres de features

In [ ]:
export_df = recipes_df.copy()
export_df['ingredientes'] = export_df['ingredientes'].apply(lambda x: ', '.join(x))
export_df = export_df.drop(columns=['ingredientes_texto'], errors='ignore')
export_df.to_csv('recipes_df.csv', index=False)
print(f'recipes_df.csv: {export_df.shape[0]} recetas, {export_df.shape[1]} columnas')

In [ ]:
scipy.sparse.save_npz('tfidf_matrix.npz', tfidf_matrix)
with open('tfidf_feature_names.json', 'w') as f:
    json.dump(list(tfidf.get_feature_names_out()), f)
print('tfidf_matrix.npz exportado')
print('tfidf_feature_names.json exportado')

In [ ]:
# Resumen de features generadas
features_df = pd.DataFrame([
    ('dificultad',           'Categórica',    'Score compuesto: ingredientes + tiempo'),
    ('prop_alacena',         'Numérica [0-1]','Proporción de ingredientes en top 60 frecuentes'),
    ('pocos_ingredientes',   'Binaria',       '1 si tiene 7 o menos ingredientes'),
    ('es_rapida',            'Binaria',       '1 si tiempo <= 35 minutos'),
    ('score_alacena',        'Numérica [0-1]','Score central: 60% alacena + 25% pocos + 15% rapidez'),
    ('*_norm',               'Numérica [0-1]','Versiones normalizadas (MinMaxScaler)'),
    ('ingredientes_texto',   'Texto',         'Ingredientes unidos para TF-IDF'),
], columns=['Feature','Tipo','Descripción'])
features_df